In [12]:
import math
import numpy as np
import random
import time
import matplotlib.pyplot as plt
from collections import defaultdict
from matplotlib.patches import Ellipse
from itertools import chain
import time
from tqdm import tqdm
from operator import attrgetter
from matplotlib.lines import Line2D
import matplotlib.colors as mcolors
from scipy.stats import chi2
from scipy.optimize import minimize
from scipy.spatial.distance import pdist
import random

WEIGHT BY MODEL RESULT ON BOUNDARY POINTS

In [13]:
# surface functions, merge function, and other helper functions

def surface_single(x, U, A, p=2, k=1):
    """
    Compute the hyperellipsoid surface function for a single point.

    Parameters
    ----------
    x : np.ndarray, shape (d,)
        The query point.
    U : np.ndarray, shape (d, d)
        Eigenvectors (columns) defining the ellipsoid orientation.
    A : np.ndarray, shape (d,)
        Semi-axis lengths along each eigenvector direction.
    p : float or np.ndarray, shape (d)
        power/exponent of the ellipsoid axes.
    k : float
        confidence value

    Returns
    -------
    float
        < 0 if x is inside the ellipsoid
        = 0 if x is on the surface
        > 0 if x is outside the ellipsoid
    """
    x_local = U.T @ x        # Project into ellipsoid's local frame, shape (d,)
    return np.sum(np.abs(x_local / A) ** p) - (k ** 2)

def surface_multi(xs, Us, As, p=2, k=1):
    """
    Compute the hyperellipsoid surface function for n points,
    each against its own ellipsoid.

    Parameters
    ----------
    xs : np.ndarray, shape (n, d)
        Query points.
    Us : np.ndarray, shape (n, d, d)
        Per-ellipsoid eigenvector matrices.
    As : np.ndarray, shape (n, d)
        Per-ellipsoid semi-axis lengths.

    Returns
    -------
    np.ndarray, shape (n,)
        Surface function value for each (point, ellipsoid) pair.
        < 0 inside, = 0 on surface, > 0 outside.
    """
    xs_local = np.einsum('ndi, ni -> nd', Us.swapaxes(1,2), xs)  # (n, d)
    return np.sum(np.abs(xs_local / As) ** p, axis=1) - (k ** 2)              # (n,)

def merge_nodes(x, y): # NOTE: New P is currently avg of the 2 that is (x.p + y.p) / 2
    new_n = x.n + y.n
    dist = x.M - y.M
    new_M = ((x.n * x.M) + (y.n * y.M)) / (new_n)
    new_S = ((x.n / new_n) * x.S) + ((y.n / new_n) * y.S) + (((x.n * y.n)/ (new_n ** 2)) * (np.outer(dist, dist)))
    eigen_value, eigen_vector = np.linalg.eigh(new_S)
    # reverse the order of the eigenvalue/vectors to be in DESCENDING order
    eigen_value = eigen_value[::-1]
    eigen_vector = eigen_vector[:, ::-1]
    # eigen_vector = eigen_vector.T
    # calculate new width that based on confidence ellipsoid
    confidence = new_n / (new_n + x.dim)
    chi = chi2.ppf(confidence, x.dim)
    new_A = np.sqrt(np.abs(eigen_value) * chi)
    new_A[new_A == 0] = x.eps

    return Node(dimension=x.dim, label=x.label, A=new_A, n=new_n, M=new_M, S=new_S, U=eigen_vector, eps=x.eps, alpha=x.alpha, p=(x.p + y.p) / 2)

def print_stat(arr, name):
    print(f"{name} Mean:", np.mean(arr))
    print(f"{name} Max:", np.max(arr))
    print(f"{name} Min:", np.min(arr))

def print_time(arr, name):
    print(f"action: {name}")
    print(f"count: {len(arr)}")
    print(f"total: {sum(arr)}")
    print(f"avg: {sum(arr)/len(arr)}")

def train_test_split(ds, test_ratio=0.2):
    ds = ds[:]  # copy
    random.shuffle(ds)

    split_idx = int(len(ds) * (1 - test_ratio))
    return ds[:split_idx], ds[split_idx:]



In [14]:
class Node():
    def __init__(self,
        dimension, label,
        A, p, eps, alpha,
        n=1,
        M=None,
        S=None,
        U=None,
        boundary=None):
        """
        A node contain information about a single ellipsoid such as its centre, cov matrix, and the axis length andd direction.
        It can be "updated" with a point to move it to learn and cover more data point.
        
        Parameters
        A (np.ndarray) (dim) : The length of each axis. Not necessary sorted.  np array 
        p (np.ndarray) (dim) : The power/exponent the nth term of the ellipsoid will be raised to when calculating surface function.
        eps (float) : A small value to add to the axis length when calculating surface function to avoid div by zero.
        alpha (float) : A hyperparameter between [0, 1] to control how the axis length is updated.
                        It specify the weight between "fixed" and "dynamic" update rule.
                        For more information see (Wongsriphisant, et al. 2026) https://doi.org/10.1016/j.eswa.2025.129818
        n (int) : The number of data points this node has learnt from.
        U (np.ndarray) (dim, dim) : The eigenvectors of the cov matrix. NOT transposed that is U[i] contain the ith eigenvector.
        M (np.ndarray) (dim) : The mean/center of the ellipsoid
        S (np.ndarray) (dim, dim) : The cov matrix of the ellipsoid. 
        boundary (dict) : A dictionary containing the boundary point and a special key, _max containing the maximum value.
        max_boundary (int) : The maximum number of points allowed in the boundary.
        """

        self.dim = dimension
        self.label = label
        self.A = A
        self.p = p
        self.eps = eps
        self.alpha = alpha
        self.n = n
        self.confidence = n / (n + self.dim)
        self.chi = chi2.ppf(self.confidence, self.dim)

        self.U = np.eye(dimension) if U is None else U
        self.M = np.array([0.] * dimension) if M is None else M
        self.S = np.zeros((dimension, dimension)) if S is None else S

        if boundary is not None:
            self.boundary = boundary
        else:
            self.boundary = []
        self.max_boundary = 2 * dimension

    def update_boundary(self, x, xd):
        """
        Update the boundary that contain the points nearest to the edge.
        This function is called when the ellipsoid is moved/updated.
        The point x is also considered whether it is a boundary point.
        """
        #print(x, self.boundary)
        # First, update the points in the boundary because the ellipsoid might have moved.
        for p in self.boundary:
            d = surface_single(p - self.M, self.U, self.A, self.p)

            if d > 0:
                for i, x in enumerate(self.boundary):
                    if np.allclose(x, p):
                        self.boundary.pop(i)
                        break

        # If the number of points in the boundary is small we simply add the new point
        d = surface_single(x - self.M, self.U, self.A, self.p)
        if d > 0:
            return None
        if len(self.boundary) < self.max_boundary:
            self.boundary.append(x)
            return None

        # Create copies of the boundary
        A = np.array(self.boundary)
        As = np.repeat(A[None, :, :], A.shape[0], axis=0)
        # Replace one row in each copy of A with x
        As[np.arange(A.shape[0]), np.arange(A.shape[0])] = x

        def f(mats):
            # mats has shape (k, n, m)
            gram = np.matmul(mats.transpose(0, 2, 1), mats)  # shape (k, m, m)
            return np.linalg.det(gram)
        det = f(np.array([A]))
        dets = f(As)

        if det > dets.max():
            pass
        else:
            self.boundary[np.argmax(dets)] = x

        """# Else we remove the point closest to the center (this might be x),        
        if d > self.boundary["_min"][1]:
            self.boundary.pop(self.boundary["_min"][0])
            self.boundary[tuple(x)] = d
            if d > self.boundary["_max"][1]:
                self.boundary["_max"] = (p, d)
            if d < self.boundary["_min"][1]:
                self.boundary["_min"] = (p, d)"""

    def update_exponent(self):
        if len(self.boundary) < self.max_boundary:
            return None
        x0 = self.p
        cons = []
        for b_point in self.boundary:
            def con(x):
                return - surface_single(b_point - self.M, self.U, self.A, x)
            cons.append({'type': 'ineq', 'fun': con})
            
        solution = minimize(np.sum, x0, method='SLSQP', constraints=cons, bounds=[(1, 10) for _ in range(len(self.p))])
        self.p = solution.x

    def update(self, x, parent):
        """
        Update the node with a new data point x.
        Performing the following steps.
        1. Calculate the new mean and cov matrix of the node including the new point
        2. Calculate the eigenvalue/vector of the new cov matrix
        3. Update the width of the axes of the ellipsoid
        4. Calculate the growth criterion to check whether the updated ellipsoid cover the new data point
        5. If the growth criterion is acceptable the node is allowed to grow/update to include this new data point,
           else the node is not updated and a new node is added to cover the new datapoint instead.

        Parameter
        x (np.ndarray) (dim) : A new data point the model need to learn/update from.
        parent (VEBF) : The VEBF model this node belong to. We only need this in case we need to add a new node to the VEBF model. 
        """

        # Calculate new mean and cov matrix
        n = self.n
        M = self.M
        alpha = n / (n + 1)
        beta = x / (n + 1)
        M_new = (alpha * M) + beta
        k_1 = (np.outer(x, x)/(n + 1)) - np.outer(M_new, M_new) + np.outer(M, M)
        k_2 = - (np.outer(M, M) / (n + 1))
        k = k_1 + k_2
        S_new = (alpha * self.S) + k

        eigen_value, eigen_vector = np.linalg.eigh(S_new)
        # reverse the order of the eigenvalue/vectors to be in DESCENDING order
        eigen_value = eigen_value[::-1]
        eigen_vector = eigen_vector[:, ::-1]

        # calculate new width with adaptively by combining fixed and dynamic width update rules
        """beta = 1 - self.alpha
        fixed = np.sqrt(np.pi * 2 * np.abs(eigen_value))
        dynamic = self.A + ((M_new - self.M) @ eigen_vector.T)
        new_A = (self.alpha * fixed) + (beta * dynamic)
        new_A[new_A == 0] = self.eps"""

        # calculate new width that based on confidence ellipsoid
        new_A = np.sqrt(np.abs(eigen_value) * self.chi)
        new_A[new_A == 0] = self.eps
        
        gc = surface_single(x - M_new, eigen_vector, self.A, self.p)
        if gc <= 0:
            # update the current node
            self.M = M_new
            self.U = eigen_vector
            self.S = S_new
            self.A = new_A
            self.n += 1
            # self.update_boundary(x, gc)
            # self.update_exponent()
            return self
        else:
            # add new node
            new_node = Node(self.dim, self.label, M=x, eps=self.eps, A=parent.default_width, alpha=self.alpha, p=self.p)
            parent.nodes[self.label].append(new_node)
            return new_node

    def __str__(self):
        return f"A node covering {self.n} points."

In [25]:
class VEBF():
    def __init__(self, dimension, merge_parameter=0, eps=0.00001, default_width=None, alpha=0.55, p=None):
        """
        The Versatile Ellipsoid Basis Function model.
        Geometrically, the model learn by covering the datapoints with ellipsoids to learn the distribution of the data.
        
        Parameter
        dimension (int) : Number of dimension of input feature.
        merge_parameter (float) : The hyperparameter that controll when two ellipsoids/nodes will be merged.
        p (np.ndarray) (dim) : The power/exponent the nth term of the ellipsoid will be raised to when calculating surface function. (dim) np array
        eps (float) : A small value to add to the axis length when calculating surface function to avoid div by zero.
        alpha (float) : A hyperparameter between [0, 1] to control how the axis length is updated.
                        It specify the weight between "fixed" and "dynamic" update rule.
                        For more information see (Wongsriphisant, et al. 2026) https://doi.org/10.1016/j.eswa.2025.129818
        default_width (np.ndarray) (dim) : The default axis width of the ellipsoid.
        """
        self.dim = dimension
        self.merge_parameter = merge_parameter
        self.nodes = {}
        self.eps = eps
        self.alpha = alpha
        self.default_width = np.array([1.] * dimension) if default_width is None else default_width
        self.p = np.array([2.] * dimension) if p is None else p # default to "normal" hyperellipsoid with p=2

    def find_shortest(self, x, y):
        # TODO: vectorize this
        min_dist = float('inf')
        min_node = None
        for node in self.nodes[y]:
            dist = np.linalg.norm(node.M - x)
            if dist < min_dist:
                min_dist = dist
                min_node = node
        return min_node

    # NOTE: can combine this with find_shortest by using y=None as default value
    def find_shortest_all(self, x, nodes=None):
        min_dist = float('inf')
        min_node = None
        nodes = self.get_nodes() if nodes is None else nodes
        for node in nodes:
            dist = np.linalg.norm(node.M - x)
            if dist < min_dist:
                min_dist = dist
                min_node = node
        return min_node

    def check_merge(self, x):
        """
        Check for pair of nodes which can be merged.
        Only consider the "latest" node, x

        Parameters:
        x (Node): The node which has just been updated or created.
        """
        nodes = self.nodes[x.label]
        if len(nodes) == 1:
            return
        nodes.remove(x)
        
        Us = np.array([node.U for node in nodes])
        Ms = np.array([node.M for node in nodes])
        As = np.array([node.A for node in nodes])
        Ps = np.array([node.p for node in nodes])
        
        scores = surface_multi(x.M - Ms, Us, As, Ps)
        merge_candidates = [node for node, score in zip(nodes, scores) if score <= self.merge_parameter]
        if len(merge_candidates) > 0:
            y = merge_candidates[0]
            nodes.remove(y)
            new_node = merge_nodes(x, y)
            nodes.append(new_node)
            self.check_merge(new_node)
            return

        n = len(nodes)
        U = x.U
        A = x.A
        P = x.p
        Us = np.broadcast_to(U, (n,) + U.shape)
        As = np.broadcast_to(A, (n,) + A.shape)
        Ps = np.broadcast_to(P, (n,) + P.shape)

        scores = surface_multi(Ms - x.M, Us, As, Ps)
        merge_candidates = [node for node, score in zip(nodes, scores) if score <= self.merge_parameter]
        if len(merge_candidates) > 0:
            y = merge_candidates[0]
            nodes.remove(y)
            new_node = merge_nodes(y, x)
            nodes.append(new_node)
            self.check_merge(new_node)
            return
            
        nodes.append(x)

    def get_nodes(self):
        return list(chain.from_iterable(self.nodes.values()))

    def min_exponent(self):
        """
        We minimize the exponent of every node,
        keeping the boundary points inside the node.
        If a node does not have enough boundary points, nothing happens.
        """
        for node in self.get_nodes():
            node.update_exponent()

    def predict(self, x):

        nodes = self.get_nodes()
        assert len(nodes) > 0, "Can't predict with empty VEBF."
        Us = np.array([node.U for node in nodes])
        Ms = np.array([node.M for node in nodes])
        As = np.array([node.A for node in nodes])
        Ps = np.array([node.p for node in nodes])

        scores = surface_multi(x - Ms, Us, As, Ps)
        idx = np.argsort(scores)
        result = nodes[idx[0]].label
        return result
        """ OLD prediction by euclidean

        candidates = [node for node, score in zip(nodes, scores) if score <= 0]

        if len(candidates) == 0:
            closest = self.find_shortest_all(x)
            return closest.label
        return self.find_shortest_all(x, candidates).label"""
            
    def train(self, x, y):
        if y not in self.nodes: # Add a new node for unseen label.
            self.nodes[y] = [Node(dimension=self.dim, label=y, M=x, eps=self.eps, A=self.default_width, alpha=self.alpha, p=self.p)]
        else: # Update closest existing node for existing label.
            shortest = self.find_shortest(x, y)
            latest = shortest.update(x, self) # latest is the node with latest change, which need to checked for merge
            self.check_merge(latest)

    def __str__(self):
        total = sum(len(x) for x in self.nodes.values())
        string = f"A vebf with {total} nodes.\n"
        for label, nodes in self.nodes.items():
            for node in nodes:
                string += f"Label {label}: {str(node)}\n"
        return string

In [26]:
class EVE():
    def __init__(self, dimension, default_width, deltas, merge_parameter=0, eps=0.00001, alpha=0.55):
        self.deltas = deltas
        self.vebfs = {}
        self.weight = {}
        self.boundary = {}
        self.max_boundary = max(100, dimension + 1)

        for delta in deltas:
            self.vebfs[delta] = VEBF(dimension=dimension, merge_parameter=merge_parameter, default_width=default_width * delta)
            self.weight[delta] = {}

    def update_boundary(self, x, y):

        if y not in self.boundary:
            self.boundary[y] = [x]
            return None

        if bool(random.getrandbits(1)):

            boundary = self.boundary[y]
            
            if len(boundary) <= self.max_boundary:
                boundary.append(x)
            else:
                idx = random.randint(0, len(boundary) - 1)
                boundary[idx] = x

    def get_boundary_point(self):
        result = []
        for label in self.boundary:
            for point in self.boundary[label]:
                result.append((point, label))
        return result

    def update_weight(self):
        points = self.get_boundary_point()
        label_count = {y: len(self.boundary[y]) for y in self.boundary}

        for delta in self.deltas:
            model = self.vebfs[delta]

            for x, y in points:
                
                if y not in self.weight[delta]:
                    self.weight[delta][y] = 0
                
                # test each VEBF on the boundary point
                y_hat = model.predict(x)
                if y_hat == y:
                    self.weight[delta][y] += 1 / label_count[y]
    
    def train(self, x, y):
        self.update_boundary(x, y)
        for vebf in self.vebfs.values():
            vebf.train(x, y)

    def predict(self, x):
        result = {}
        for delta in self.deltas:
            pred = self.vebfs[delta].predict(x)
            if pred in result:
                result[pred] += self.weight[delta][pred]
            else:
                result[pred] = self.weight[delta][pred]
        return max(result, key=result.get)
    

In [17]:
# Loading the datasets

def train_test_split(ds, test_ratio=0.2):
    ds = ds[:]  # copy
    random.shuffle(ds)

    split_idx = int(len(ds) * (1 - test_ratio))
    return ds[:split_idx], ds[split_idx:]
    
with open("data/iris/iris.data", "r") as f:
    iris_data = f.readlines()
iris_ds = []
for line in iris_data[:-1]:
    line = line.strip().split(',')
    x = np.array(line[:-1], dtype=float)
    y = line[-1]
    iris_ds.append((x, y))

with open("data/ecoli/ecoli.data", "r") as f:
    ecoli_data = f.readlines()
ecoli_ds = []
for line in ecoli_data:
    line = line.strip().split()
    x = np.array(line[1:-1], dtype=float)
    y = line[-1]
    ecoli_ds.append((x, y))

with open("data/image_seg/segmentation.data", "r") as f:
    seg_train_data = f.readlines()
with open("data/image_seg/segmentation.test", "r") as f:
    seg_test_data = f.readlines()
seg_train = []
for line in seg_train_data[5:]:
    line = line.strip().split(",")
    y = line[0]
    x = np.array(line[1:], dtype=float)
    seg_train.append((x, y))
seg_test = []
for line in seg_test_data[5:]:
    line = line.strip().split(",")
    y = line[0]
    x = np.array(line[1:], dtype=float)
    seg_test.append((x, y))

with open("data/waveform/waveform.data", "r") as f:
    waveform_data = f.readlines()
wave_ds = []
for line in waveform_data:
    line = line.strip().split(",")
    y = line[-1]
    x = np.array(line[:-1], dtype=float)
    wave_ds.append((x, y))

with open("data/yeast/yeast.data", "r") as f:
    yeast_data = f.readlines()
yeast_ds = []
for line in yeast_data:
    line = line.strip().split()
    x = np.array(line[1:-1], dtype=float)
    y = line[-1]
    yeast_ds.append((x, y))

with open("data/anuran/Frogs_MFCCs.csv", "r") as f:
    anuran_data = f.readlines()
anuran_ds = []
for line in anuran_data[1:]:
    line = line.strip().split(",")
    x = np.array(line[:-4], dtype=float)
    y = tuple(line[-4:-1])
    anuran_ds.append((x, y))

with open("data/spambase/spambase.data", "r") as f:
    spam_data = f.readlines()
spam_ds = []
for line in spam_data:
    line = line.strip().split(",")
    x = np.array(line[:-1], dtype=float)
    y = line[-1]
    spam_ds.append((x, y))

with open("data/letter/letter-recognition.data", "r") as f:
    letter_data = f.readlines()
letter_ds = []
for line in letter_data:
    line = line.strip().split(",")
    x = np.array(line[1:], dtype=float)
    y = line[0]
    letter_ds.append((x, y))

with open("data/bankrupt/data.csv", "r") as f:
    bankrupt_data = f.readlines()
bankrupt_ds = []
for line in bankrupt_data[1:]:
    line = line.strip().split(",")
    x = np.array(line[1:], dtype=float)
    y = line[0]
    bankrupt_ds.append((x, y))

with open("data/digits/optdigits.tes", "r") as f:
    digits_test_data = f.readlines()
digits_test = []
for line in digits_test_data[:]:
    line = line.strip().split(",")
    x = np.array(line[:-1], dtype=float)
    y = line[-1]
    digits_test.append((x, y))

with open("data/digits/optdigits.tra", "r") as f:
    digits_train_data = f.readlines()
digits_train = []
for line in digits_train_data[:]:
    line = line.strip().split(",")
    x = np.array(line[:-1], dtype=float)
    y = line[-1]
    digits_train.append((x, y))

with open("data/phishing/PhishingData.arff", "r") as f:
    phishing_data = f.readlines()
phishing_ds = []
for line in phishing_data[14:]:
    line = line.strip().split(",")
    x = np.array(line[:-1], dtype=float)
    y = line[-1]
    phishing_ds.append((x, y))


In [18]:
def initial_width(ds, delta=1):
    avg_dist = pdist([x for x,_ in ds], metric="euclidean").mean()
    return avg_dist

iris_dim = len(iris_ds[0][0])
anuran_dim = len(anuran_ds[0][0])
spam_dim = len(spam_ds[0][0])
seg_dim = len(seg_train[0][0])
wave_dim = len(wave_ds[0][0])
letter_dim = len(letter_ds[0][0])
yeast_dim = len(yeast_ds[0][0])
digits_dim = len(digits_train[0][0])
bankrupt_dim = len(bankrupt_ds[0][0])
phishing_dim = len(phishing_ds[0][0])

iris_A = initial_width(iris_ds)
anuran_A = initial_width(anuran_ds)
spam_A = initial_width(spam_ds)
seg_A = initial_width(seg_train)
wave_A = initial_width(wave_ds)
letter_A = initial_width(letter_ds)
yeast_A = initial_width(yeast_ds)
digits_A = initial_width(digits_train)
bankrupt_A = initial_width(bankrupt_ds)
phishing_A = initial_width(phishing_ds)
# The implementation of confidence based VEBF takes array of width as initial width
iris_A = np.array([iris_A] * iris_dim)
anuran_A = np.array([anuran_A] * anuran_dim)
spam_A = np.array([spam_A] * spam_dim)
seg_A = np.array([seg_A] * seg_dim)
wave_A = np.array([wave_A] * wave_dim)
letter_A = np.array([letter_A] * letter_dim)
yeast_A = np.array([yeast_A] * yeast_dim)
digits_A = np.array([digits_A] * digits_dim)
bankrupt_A = np.array([bankrupt_A] * bankrupt_dim)
phishing_A = np.array([phishing_A] * phishing_dim)




In [19]:
def print_stat(arr, name):
    print(f"{name} Mean:", np.mean(arr))
    print(f"{name} Max:", np.max(arr))
    print(f"{name} Min:", np.min(arr))

In [27]:
anuran_result = []
deltas = [2**-1, 2**-0, 2**1, 2**2, 2**3]
for i in tqdm(range(10)):

    # shuffle the train/test split
    anuran_train, anuran_test = train_test_split(anuran_ds)
    model = EVE(dimension=anuran_dim, merge_parameter=0., default_width=anuran_A, deltas=deltas)
    for idx in range(len(anuran_train)):
        x,y = anuran_train[idx]
        x = np.array(x,dtype=float)
        model.train(x, y)

    model.update_weight()

    # test the model
    correct = 0
    for x, label in anuran_test:
        pred = model.predict(x)
        if pred == label:
            correct += 1
    
    anuran_result.append(correct/len(anuran_test))
print_stat(anuran_result, "anuran")
print()

100%|██████████| 10/10 [00:57<00:00,  5.73s/it]

anuran Mean: 0.9687977762334956
anuran Max: 0.9777623349548298
anuran Min: 0.9617790132036136



In [28]:
spam_result = []
deltas = [2**-1, 2**-0, 2**1, 2**2, 2**3]
for i in tqdm(range(5)):

    # shuffle the train/test split
    spam_train, spam_test = train_test_split(spam_ds)
    model = EVE(dimension=spam_dim, merge_parameter=0., default_width=spam_A, deltas=deltas)
    for idx in range(len(spam_train)):
        x,y = spam_train[idx]
        x = np.array(x,dtype=float)
        model.train(x, y)

    model.update_weight()

    # test the model
    correct = 0
    for x, label in spam_test:
        pred = model.predict(x)
        if pred == label:
            correct += 1
    
    spam_result.append(correct/len(spam_test))
print_stat(spam_result, "spam")

100%|██████████| 5/5 [02:02<00:00, 24.46s/it]

spam Mean: 0.9022801302931596
spam Max: 0.9207383279044516
spam Min: 0.8881650380021715


In [29]:
letter_result = []
deltas = [2**-1, 2**-0, 2**1, 2**2, 2**3]
for i in tqdm(range(5)):

    # shuffle the train/test split
    letter_train, letter_test = train_test_split(letter_ds)
    model = EVE(dimension=letter_dim, merge_parameter=0., default_width=letter_A, deltas=deltas)
    for idx in range(len(letter_train)):
        x,y = letter_train[idx]
        x = np.array(x,dtype=float)
        model.train(x, y)

    model.update_weight()

    # test the model
    correct = 0
    for x, label in letter_test:
        pred = model.predict(x)
        if pred == label:
            correct += 1
    
    letter_result.append(correct/len(letter_test))
print_stat(letter_result, "letter")

100%|██████████| 5/5 [01:10<00:00, 14.18s/it]

letter Mean: 0.8748000000000001
letter Max: 0.881
letter Min: 0.868


In [30]:
seg_result = []
deltas = [2**-1, 2**-0, 2**1, 2**2, 2**3]
for i in tqdm(range(30)):

    # shuffle the train/test split
    random.shuffle(seg_train)
    model = EVE(dimension=seg_dim, merge_parameter=0., default_width=seg_A, deltas=deltas)
    for idx in range(len(seg_train)):
        x,y = seg_train[idx]
        x = np.array(x,dtype=float)
        model.train(x, y)

    model.update_weight()

    # test the model
    correct = 0
    for x, label in seg_test:
        pred = model.predict(x)
        if pred == label:
            correct += 1
    
    seg_result.append(correct/len(seg_test))
print_stat(seg_result, "seg")

100%|██████████| 30/30 [00:17<00:00,  1.69it/s]

seg Mean: 0.7562222222222221
seg Max: 0.8419047619047619
seg Min: 0.6042857142857143


In [31]:
digits_result = []
deltas = [2**-1, 2**-0, 2**1, 2**2, 2**3]
for i in tqdm(range(5)):

    # shuffle the train/test split
    random.shuffle(digits_train)
    model = EVE(dimension=digits_dim, merge_parameter=0., default_width=digits_A, deltas=deltas)
    for idx in range(len(digits_train)):
        x,y = digits_train[idx]
        x = np.array(x,dtype=float)
        model.train(x, y)

    model.update_weight()

    # test the model
    correct = 0
    for x, label in digits_test:
        pred = model.predict(x)
        if pred == label:
            correct += 1
    
    digits_result.append(correct/len(digits_test))
print_stat(digits_result, "digit")

100%|██████████| 5/5 [03:13<00:00, 38.73s/it]

digit Mean: 0.9035058430717864
digit Max: 0.9532554257095158
digit Min: 0.8503060656649972


In [32]:
phishing_result = []
deltas = [2**-1, 2**-0, 2**1, 2**2, 2**3]
for i in tqdm(range(5)):

    # shuffle the train/test split
    phishing_train, phishing_test = train_test_split(phishing_ds)
    model = EVE(dimension=phishing_dim, merge_parameter=0., default_width=phishing_A, deltas=deltas)
    for idx in range(len(phishing_train)):
        x,y = phishing_train[idx]
        x = np.array(x,dtype=float)
        model.train(x, y)

    model.update_weight()

    # test the model
    correct = 0
    for x, label in phishing_test:
        pred = model.predict(x)
        if pred == label:
            correct += 1
    
    phishing_result.append(correct/len(phishing_test))
print_stat(phishing_result, "phishing")

100%|██████████| 5/5 [00:03<00:00,  1.40it/s]

phishing Mean: 0.825830258302583
phishing Max: 0.8597785977859779
phishing Min: 0.8007380073800738


In [33]:
bankrupt_result = []
deltas = [2**-1, 2**-0, 2**1, 2**2, 2**3]
for i in tqdm(range(5)):

    # shuffle the train/test split
    bankrupt_train, bankrupt_test = train_test_split(bankrupt_ds)
    model = EVE(dimension=bankrupt_dim, merge_parameter=0., default_width=bankrupt_A, deltas=deltas)
    for idx in range(len(bankrupt_train)):
        x,y = bankrupt_train[idx]
        x = np.array(x,dtype=float)
        model.train(x, y)

    model.update_weight()

    # test the model
    correct = 0
    for x, label in bankrupt_test:
        pred = model.predict(x)
        if pred == label:
            correct += 1
    
    bankrupt_result.append(correct/len(bankrupt_test))
print_stat(bankrupt_result, "bankrupt")

100%|██████████| 5/5 [08:42<00:00, 104.42s/it]

bankrupt Mean: 0.9626099706744868
bankrupt Max: 0.9655425219941349
bankrupt Min: 0.9582111436950147
